In [29]:
import warnings
warnings.filterwarnings("ignore")

from copy import deepcopy
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision import models, transforms
from torchvision.transforms.functional import InterpolationMode
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [30]:
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 8
NUM_CLASSES = 2

image_paths = {
    "train": '../data/preprocessed_train_images',
    "test": '../data/preprocessed_test_images',
    "validation": '../data/preprocessed_validation_images'
}

label_dfs = {
    "train": pd.read_csv('../data/2 Classes Label/train_1.csv'),
    "test": pd.read_csv('../data/2 Classes Label/test.csv'),
    "validation": pd.read_csv('../data/2 Classes Label/valid.csv')
}

for key, df in label_dfs.items():
    df.sort_values(by='id_code', inplace=True)
    df['image_path'] = df['id_code'].apply(lambda x: f"{image_paths[key]}/{x}.png")

train_labels_df = label_dfs['train']
test_labels_df = label_dfs['test']
validation_labels_df = label_dfs['validation']

In [31]:
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE, interpolation=InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=30, interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [32]:
class CustomImageDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

In [33]:
train_dataset = CustomImageDataset(
    image_paths=train_labels_df['image_path'].tolist(),
    labels=train_labels_df['diagnosis'].tolist(),
    transform=transform
)

test_dataset = CustomImageDataset(
    image_paths=test_labels_df['image_path'].tolist(),
    labels=test_labels_df['diagnosis'].tolist(),
    transform=transform
)

validation_dataset = CustomImageDataset(
    image_paths=validation_labels_df['image_path'].tolist(),
    labels=validation_labels_df['diagnosis'].tolist(),
    transform=transform
)

In [34]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
val_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [35]:
class CNNModel(nn.Module):
    def __init__(self, num_classes=2):
        super(CNNModel, self).__init__()
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv_block4 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv_block5 = nn.Sequential(
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.fc_block1 = nn.Sequential(
            nn.Linear(512 * 8 * 8, 1024),
            nn.ReLU(),
            nn.Dropout(p=0.5)
        )
        self.fc_middle = nn.Sequential(
            nn.Linear(1024, 768),
            nn.ReLU(),
            nn.Dropout(p=0.5)
        )
        self.fc_block2 = nn.Sequential(
            nn.Linear(1024, num_classes),
            nn.ReLU(),
            nn.Dropout(p=0.5)
        )
        self.fc_block3 = nn.Sequential(
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        x = self.conv_block5(x)
        x = x.view(x.size(0), -1)
        x = self.fc_block1(x)
        x = self.fc_middle(x)
        x = self.fc_block2(x)
        x = self.fc_block3(x)
        return x

In [36]:
model = CNNModel(num_classes=NUM_CLASSES)

criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

In [37]:
def train_model(
    model, 
    train_loader, 
    val_loader, 
    criterion, 
    optimizer, 
    epochs=30, 
    dry_run=False, 
    save_path="best_model.pth"
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": []}
    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_train_loss = float("inf")
    best_train_acc = 0.0

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        
        model.train()
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct_predictions += torch.sum(preds == labels).item()
            total_samples += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_accuracy = correct_predictions / total_samples
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)

        # Update best training loss and accuracy
        if train_loss < best_train_loss:
            best_train_loss = train_loss
        if train_accuracy > best_train_acc:
            best_train_acc = train_accuracy

        model.eval()
        val_loss = 0.0
        correct_predictions = 0
        total_samples = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, preds = torch.max(outputs, 1)
                correct_predictions += torch.sum(preds == labels).item()
                total_samples += labels.size(0)

        val_loss /= len(val_loader)
        val_accuracy = correct_predictions / total_samples
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_accuracy
            print(f"Validation loss improved. Saving model to {save_path}...")
            torch.save(model.state_dict(), save_path)

        if dry_run and epoch == 0:
            print("Dry-run mode activated. Training stopped after 1 epoch.")
            break

    print(f"Training completed. "
          f"Best Train Loss: {best_train_loss:.4f}, Best Train Acc: {best_train_acc:.4f}, "
          f"Best Val Loss: {best_val_loss:.4f}, Best Val Acc: {best_val_acc:.4f}")
    
    return model, history


In [ ]:
def display_classification_report_table(report, num_classes):
    class_metrics = {f"Class {i}": report[str(i)] for i in range(num_classes)}
    df_report = pd.DataFrame(class_metrics).T[['precision', 'recall', 'f1-score', 'support']]
    
    print("\nClassification Report:")
    print(df_report)
    
    fig, ax = plt.subplots(figsize=(8, num_classes * 0.5 + 1))
    ax.axis('tight')
    ax.axis('off')
    table = ax.table(cellText=df_report.values, colLabels=df_report.columns, rowLabels=df_report.index, loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.auto_set_column_width(col=list(range(len(df_report.columns))))
    plt.title("Classification Report")
    plt.show()

In [38]:
def evaluate_model(model, data_loader, criterion, num_classes):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            correct_predictions += torch.sum(preds == labels).item()
            total_samples += labels.size(0)

    val_accuracy = correct_predictions / total_samples
    val_loss = total_loss / len(data_loader)

    print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

    cm = confusion_matrix(all_labels, all_preds, labels=range(num_classes))
    report = classification_report(all_labels, all_preds, labels=range(num_classes), output_dict=True)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(num_classes))
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.show()

    display_classification_report_table(report, num_classes)

    return val_loss, val_accuracy

In [39]:
def plot_training_history(history):
    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('Training History', fontsize=16)

    axes[0, 0].plot(epochs, history['train_loss'], label='Train Loss', marker='o')
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].set_xlabel('Epochs')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].grid(True)
    axes[0, 0].legend()

    axes[0, 1].plot(epochs, history['val_loss'], label='Validation Loss', marker='o', color='orange')
    axes[0, 1].set_title('Validation Loss')
    axes[0, 1].set_xlabel('Epochs')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].grid(True)
    axes[0, 1].legend()

    axes[1, 0].plot(epochs, history['train_accuracy'], label='Train Accuracy', marker='o', color='green')
    axes[1, 0].set_title('Training Accuracy')
    axes[1, 0].set_xlabel('Epochs')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].grid(True)
    axes[1, 0].legend()

    axes[1, 1].plot(epochs, history['val_accuracy'], label='Validation Accuracy', marker='o', color='red')
    axes[1, 1].set_title('Validation Accuracy')
    axes[1, 1].set_xlabel('Epochs')
    axes[1, 1].set_ylabel('Accuracy')
    axes[1, 1].grid(True)
    axes[1, 1].legend()

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


In [41]:
def load_model(model, optimizer, path="model_checkpoint.pth"):
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print(f"Model and optimizer states loaded from {path}")
    return model, optimizer

In [ ]:
trained_model, history = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=30)

In [ ]:
plot_training_history(history)

In [ ]:
val_loss, val_accuracy = evaluate_model(model, test_loader, '/kaggle/working/best_model.pth')